# Lezione 1: Il Motore del Flow Matching
**File**: `cfm/flow_matcher.py`  
**Ruolo nel progetto**: È il cuore matematico dell'intero sistema. Tutto il resto — la rete neurale, il grafo biologico, la maschera — serve a implementare *una singola funzione* che questo file chiede: $v_\theta(x_t, t, c)$.

---

## 0. Setup e Import
Prima di tutto, assicuriamoci di poter importare i moduli del progetto.

In [ ]:
import sys, os
# Aggiungiamo la root del progetto al path per importare i nostri moduli
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment

sns.set_theme(style='whitegrid')
torch.manual_seed(42)
print(f'PyTorch: {torch.__version__}')

---
## 1. Il Problema Biologico: Perché Serve un Modello Generativo?

Immagina di avere due popolazioni di cellule:
- **Controllo** ($x_0$): cellule normali, non perturbate
- **Perturbate** ($x_1$): cellule in cui abbiamo silenziato un gene (es. CRISPRi knockout)

Ogni cellula è un vettore di $G$ valori di espressione genica: $x \in \mathbb{R}^G$.

**Problema**: dato un *nuovo* gene mai perturbato prima, come prevediamo la distribuzione delle cellule perturbate?

Ci serve un modello che apprenda a **trasformare** la distribuzione di controllo $p_{\text{ctrl}}$ nella distribuzione perturbata $p_{\text{pert}}$.

### Creiamo un esempio giocattolo per visualizzare il concetto

In [ ]:
# Simuliamo due popolazioni di cellule in 2D (per poterle visualizzare)
# In realtà G = migliaia di geni, ma il principio è identico

N = 200  # numero di cellule per condizione

# Popolazione di controllo: centrata in (-2, -1)
x_0_pop = torch.randn(N, 2) * 0.4 + torch.tensor([-2.0, -1.0])

# Popolazione perturbata: centrata in (+2, +1.5), forma leggermente diversa
x_1_pop = torch.randn(N, 2) * 0.5 + torch.tensor([+2.0, +1.5])

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x_0_pop[:, 0], x_0_pop[:, 1], c='royalblue', alpha=0.6, s=30, label='Controllo $x_0$')
ax.scatter(x_1_pop[:, 0], x_1_pop[:, 1], c='crimson', alpha=0.6, s=30, label='Perturbato $x_1$')
ax.set_xlabel('Gene 1 (espressione)', fontsize=12)
ax.set_ylabel('Gene 2 (espressione)', fontsize=12)
ax.set_title('Il Problema: come trasformare la nuvola blu nella nuvola rossa?', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 2. L'Idea del Flow Matching: Un Campo Vettoriale Continuo

Il **Flow Matching** dice: definiamo una trasformazione *continua* nel tempo $t \in [0, 1]$ tale che:
- A $t = 0$, abbiamo la distribuzione di controllo
- A $t = 1$, abbiamo la distribuzione perturbata
- Per ogni $t$ intermedio, abbiamo una distribuzione intermedia

Questa trasformazione è governata da un **campo vettoriale** $v(x, t)$:

$$\frac{dx_t}{dt} = v_\theta(x_t, t, c)$$

È un'**equazione differenziale ordinaria** (ODE): la velocità di spostamento di ogni punto $x_t$ nello spazio è determinata dalla rete neurale $v_\theta$.

### Visualizziamo l'interpolazione lineare

In [ ]:
# Prendiamo un sottoinsieme di cellule per visualizzare le traiettorie
n_show = 15
x0_show = x_0_pop[:n_show]
x1_show = x_1_pop[:n_show]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, t_val in enumerate([0.0, 0.33, 0.66, 1.0]):
    ax = axes[idx]
    # Interpolazione lineare: mu_t = (1 - t) * x_0 + t * x_1
    x_t = (1 - t_val) * x0_show + t_val * x1_show
    
    # Sfondo: tutte le cellule di controllo e perturbate
    ax.scatter(x_0_pop[:, 0], x_0_pop[:, 1], c='royalblue', alpha=0.15, s=10)
    ax.scatter(x_1_pop[:, 0], x_1_pop[:, 1], c='crimson', alpha=0.15, s=10)
    # Cellule interpolate
    ax.scatter(x_t[:, 0], x_t[:, 1], c='darkorange', s=50, zorder=5, edgecolors='black', linewidths=0.5)
    
    ax.set_title(f't = {t_val:.2f}', fontsize=14, fontweight='bold')
    ax.set_xlim(-4, 4)
    ax.set_ylim(-3, 4)
    ax.set_xlabel('Gene 1')
    ax.set_ylabel('Gene 2')

plt.suptitle('Interpolazione lineare: $x_t = (1-t) \cdot x_0 + t \cdot x_1$', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Le cellule arancioni si spostano linearmente dal controllo (blu) alla perturbazione (rosso).')
print('Questa è la traiettoria IDEALE che il Flow Matching cerca di apprendere.')

---
## 3. Scriviamo il Codice Pezzo per Pezzo

Ora ricostruiamo `flow_matcher.py` dal nulla, spiegando ogni riga.

### 3.1 Il Costruttore: `sigma_min`

L'unico iperparametro della classe è `sigma_min`: la "larghezza del tubo" gaussiano attorno alla traiettoria rettilinea.

- Se `sigma_min = 0`: il percorso è deterministico (segmento perfetto)
- Se `sigma_min > 0`: aggiungiamo una piccola perturbazione gaussiana che **regolarizza** l'apprendimento

In [ ]:
class ConditionalFlowMatcher:
    """
    Motore base di Conditional Flow Matching.
    
    Dati x_0 (controllo) e x_1 (perturbato), il probability path è:
        p_t(x | x_0, x_1) = N(x | mu_t, sigma_min^2 * I)
    dove:
        mu_t = (1 - t) * x_0 + t * x_1
    
    Il campo vettoriale condizionato target è:
        u_t = d/dt mu_t = x_1 - x_0
    """
    
    def __init__(self, sigma_min: float = 1e-4):
        self.sigma_min = sigma_min

# Creiamo un'istanza
cfm = ConditionalFlowMatcher(sigma_min=0.01)
print(f'CFM creato con sigma_min = {cfm.sigma_min}')

### 3.2 Campionamento del Tempo: `sample_time`

Campioniamo $t \sim \text{Uniform}(0, 1)$ **indipendentemente** per ogni cellula nel batch.

**Perché la forma è `[B, 1]` e non `[B]`?**  
Per il **broadcasting** di PyTorch: $t$ deve moltiplicarsi con tensori di forma `[B, G]`. La dimensione extra `1` si espande automaticamente lungo i $G$ geni:

```
t:   [B, 1] → broadcast → [B, G]
x_0: [B, G]
```

In [ ]:
def sample_time(batch_size, device):
    """Campiona t ~ Uniform(0, 1) per ogni cellula nel batch."""
    return torch.rand(batch_size, 1, device=device)

# Aggiungiamo il metodo alla classe
ConditionalFlowMatcher.sample_time = sample_time

# Proviamolo
t = cfm.sample_time(batch_size=8, device='cpu')
print(f'Forma di t: {t.shape}')  # [8, 1]
print(f'Valori (ogni cellula ha il suo tempo):') 
print(t.squeeze().numpy().round(3))

In [ ]:
# Verifichiamo il broadcasting: t [B, 1] * x_0 [B, G]
B, G = 4, 5
t_demo = torch.tensor([[0.2], [0.5], [0.8], [1.0]])  # [4, 1]
x_demo = torch.ones(B, G)  # [4, 5]

result = t_demo * x_demo
print(f't shape: {t_demo.shape}')
print(f'x shape: {x_demo.shape}')
print(f'Risultato shape: {result.shape}')  # [4, 5]
print(f'\nRisultato (ogni riga è moltiplicata per il suo t):')
print(result)
print('\n✓ Il broadcasting espande t lungo la dimensione dei geni automaticamente!')

### 3.3 Il Cuore: `sample_location_and_target`

Questa è **la funzione più importante** di tutto il file. Fa tre cose:

1. **Campiona il tempo** $t \sim U(0, 1)$
2. **Interpola** tra $x_0$ e $x_1$: $\quad x_t = (1-t) \cdot x_0 + t \cdot x_1 + \sigma_{\min} \cdot \epsilon$
3. **Calcola il target analitico**: $\quad u_t = x_1 - x_0$

#### Perché $u_t = x_1 - x_0$ è il target?

La media del probability path è $\mu_t = (1-t) \cdot x_0 + t \cdot x_1$.

La sua derivata rispetto a $t$ è:

$$\frac{d\mu_t}{dt} = \frac{d}{dt}\Big[(1-t) \cdot x_0 + t \cdot x_1\Big] = -x_0 + x_1 = x_1 - x_0$$

**Il target è costante nel tempo!** Non dipende da $t$ né da $x_t$. Questo rende l'addestramento una semplice regressione.

In [ ]:
def sample_location_and_target(self, x_0, x_1, t=None):
    """
    Interpola tra x_0 e x_1 al tempo t.
    
    Returns:
        t:   Tempi campionati [B, 1]
        x_t: Punti interpolati [B, D]
        u_t: Vettore velocità target [B, D]
    """
    batch_size = x_0.shape[0]
    device = x_0.device

    # Passo 1: campiona t se non fornito
    if t is None:
        t = self.sample_time(batch_size, device)

    # Passo 2: interpolazione lineare
    mu_t = (1.0 - t) * x_0 + t * x_1

    # Passo 3: aggiunta di rumore regolarizzante
    if self.sigma_min > 0:
        noise = torch.randn_like(x_0)  # epsilon ~ N(0, I)
        x_t = mu_t + self.sigma_min * noise
    else:
        x_t = mu_t

    # Passo 4: target analitico (derivata di mu_t rispetto a t)
    u_t = x_1 - x_0

    return t, x_t, u_t

ConditionalFlowMatcher.sample_location_and_target = sample_location_and_target
print('✓ sample_location_and_target definito')

#### Verifichiamo gli estremi: a $t=0$ deve dare $x_0$, a $t=1$ deve dare $x_1$

In [ ]:
cfm_test = ConditionalFlowMatcher(sigma_min=0.0)  # sigma=0 per test esatto

x_0 = torch.tensor([[1.0, 2.0, 3.0]])
x_1 = torch.tensor([[5.0, 6.0, 7.0]])

# Test a t=0: deve restituire x_0
t_zero = torch.tensor([[0.0]])
_, x_t, u_t = cfm_test.sample_location_and_target(x_0, x_1, t=t_zero)
print(f't=0:  x_t = {x_t.numpy()} (atteso: {x_0.numpy()})')
assert torch.allclose(x_t, x_0), 'ERRORE!'

# Test a t=1: deve restituire x_1
t_one = torch.tensor([[1.0]])
_, x_t, u_t = cfm_test.sample_location_and_target(x_0, x_1, t=t_one)
print(f't=1:  x_t = {x_t.numpy()} (atteso: {x_1.numpy()})')
assert torch.allclose(x_t, x_1), 'ERRORE!'

# Il target è sempre lo stesso, indipendentemente da t
print(f'\nu_t (target) = {u_t.numpy()} = x_1 - x_0 = {(x_1 - x_0).numpy()}')
print('\n✓ Gli estremi sono corretti e il target è costante!')

#### Visualizziamo l'effetto di `sigma_min`

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x_0_test = torch.tensor([[-2.0, -1.0]]).expand(300, -1)  # 300 copie della stessa cellula
x_1_test = torch.tensor([[+2.0, +1.5]]).expand(300, -1)

for idx, (sigma, title) in enumerate([
    (0.0, '$\sigma_{\min} = 0$ (deterministico)'),
    (0.05, '$\sigma_{\min} = 0.05$ (regolarizzato)'),
    (0.3, '$\sigma_{\min} = 0.3$ (troppo rumore)'),
]):
    ax = axes[idx]
    cfm_vis = ConditionalFlowMatcher(sigma_min=sigma)
    
    t_fixed = torch.full((300, 1), 0.5)  # tutti a t=0.5
    _, x_t, _ = cfm_vis.sample_location_and_target(x_0_test, x_1_test, t=t_fixed)
    
    # Punto medio esatto (senza rumore)
    midpoint = 0.5 * x_0_test[0] + 0.5 * x_1_test[0]
    
    ax.scatter(x_t[:, 0].numpy(), x_t[:, 1].numpy(), c='darkorange', alpha=0.4, s=15)
    ax.scatter(*midpoint.numpy(), c='black', s=100, marker='x', zorder=5, label='Centro esatto $\mu_t$')
    ax.scatter(-2, -1, c='royalblue', s=100, marker='o', zorder=5, label='$x_0$')
    ax.scatter(+2, +1.5, c='crimson', s=100, marker='o', zorder=5, label='$x_1$')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3, 3.5)
    ax.legend(fontsize=9)

plt.suptitle('Effetto di $\sigma_{\min}$ sul probability path a $t = 0.5$', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('σ=0: tutti i punti sovrapposti al centro. Nessuna regolarizzazione.')
print('σ=0.05: leggera nuvola attorno al centro. Il modello vede punti vicini alla traiettoria.')
print('σ=0.3: troppo rumore! Il modello non distingue più dove sta la traiettoria.')

### 3.4 La Loss di Addestramento: `compute_loss`

Ora che abbiamo $x_t$ e $u_t$, la rete neurale $v_\theta$ deve predire la velocità e pagarla con una **regressione L2**:

$$\mathcal{L}_{\text{CFM}}(\theta) = \frac{1}{B \cdot G} \sum_{i=1}^B \sum_{g=1}^G \Big( v_{\theta,g}(x_{t,i}, t_i, c_i) - u_{t,i,g} \Big)^2$$

Il flusso completo di un singolo step di training:

```
(x_0, x_1)  →  sample_location_and_target  →  (x_t, t, u_t)
                                                    ↓
                                            v_θ(x_t, t, c)  →  v_pred
                                                    ↓
                                            Loss = mean(‖v_pred - u_t‖²)
                                                    ↓
                                            loss.backward() → optimizer.step()
```

In [ ]:
def compute_loss(self, vector_field_model, x_0, x_1, condition=None):
    """
    Calcola la loss di Flow Matching L2.
    
    Args:
        vector_field_model: rete v_theta(x, t, condition) → velocità
        x_0: profili di controllo [B, G]
        x_1: profili perturbati [B, G]
        condition: informazione sulla perturbazione [B, C]
    """
    # 1. Campiona il punto intermedio x_t e il target u_t
    t, x_t, u_t = self.sample_location_and_target(x_0, x_1)

    # 2. La rete predice la velocità in quel punto
    v_pred = vector_field_model(x_t, t, condition)

    # 3. Regressione L2: quanto sbaglia la predizione?
    loss = torch.mean((v_pred - u_t) ** 2)

    # 4. Metriche diagnostiche (per il logging, non per i gradienti)
    metrics = {
        'loss_cfm': loss.item(),
        'target_norm': torch.norm(u_t, dim=-1).mean().item(),
        'pred_norm': torch.norm(v_pred, dim=-1).mean().item(),
    }

    return loss, metrics

ConditionalFlowMatcher.compute_loss = compute_loss
print('✓ compute_loss definito')

#### Proviamola con una rete giocattolo!

In [ ]:
# Rete giocattolo: MLP che riceve [x, t] e predice la velocità
class ToyVectorField(nn.Module):
    def __init__(self, dim=2, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, hidden),  # input: [x (2), t (1)] = 3
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, dim),       # output: velocità [2]
        )
    
    def forward(self, x, t, condition=None):
        return self.net(torch.cat([x, t], dim=-1))

model = ToyVectorField(dim=2)
cfm = ConditionalFlowMatcher(sigma_min=0.01)

# Un batch di 16 coppie (controllo, perturbato)
x0 = torch.randn(16, 2) + torch.tensor([-2.0, -1.0])
x1 = torch.randn(16, 2) + torch.tensor([+2.0, +1.5])

loss, metrics = cfm.compute_loss(model, x0, x1)
print(f'Loss iniziale (rete casuale): {loss.item():.4f}')
print(f'Norma del target u_t: {metrics["target_norm"]:.4f}')
print(f'Norma della predizione v_θ: {metrics["pred_norm"]:.4f}')
print('\nLa loss è alta perché la rete non ha ancora imparato nulla!')

---
## 4. Addestriamo la Rete Giocattolo!

Verifichiamo che il Flow Matching impari a traslare una gaussiana 2D.

In [ ]:
torch.manual_seed(42)
model = ToyVectorField(dim=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
cfm = ConditionalFlowMatcher(sigma_min=0.01)

mean_0 = torch.tensor([-2.0, -1.0])
mean_1 = torch.tensor([+2.0, +1.5])

losses = []
for step in range(300):
    optimizer.zero_grad()
    x0 = mean_0 + 0.3 * torch.randn(128, 2)
    x1 = mean_1 + 0.3 * torch.randn(128, 2)
    
    loss, _ = cfm.compute_loss(model, x0, x1)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

# Plot della curva di apprendimento
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, color='teal', linewidth=1.5)
ax.set_xlabel('Step di training', fontsize=12)
ax.set_ylabel('CFM Loss $\\|v_\\theta - u_t\\|^2$', fontsize=12)
ax.set_title('Curva di apprendimento del Flow Matching', fontsize=13, fontweight='bold')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f'Loss iniziale: {losses[0]:.4f}')
print(f'Loss finale:   {losses[-1]:.4f}')
print(f'Riduzione: {losses[0]/losses[-1]:.0f}x')

#### Visualizziamo il campo vettoriale appreso

Ora plottiamo le frecce che la rete ha imparato nello spazio 2D.

In [ ]:
model.eval()
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for idx, t_val in enumerate([0.0, 0.5, 0.9]):
    ax = axes[idx]
    
    # Griglia di punti per visualizzare il campo
    xx = np.linspace(-4, 4, 15)
    yy = np.linspace(-3, 4, 12)
    X, Y = np.meshgrid(xx, yy)
    points = torch.tensor(np.stack([X.ravel(), Y.ravel()], axis=-1), dtype=torch.float32)
    t_grid = torch.full((points.shape[0], 1), t_val)
    
    with torch.no_grad():
        v = model(points, t_grid).numpy()
    
    # Frecce del campo vettoriale
    ax.quiver(X.ravel(), Y.ravel(), v[:, 0], v[:, 1], 
              color='dimgray', alpha=0.7, scale=50)
    
    # Distribuzioni sorgente e target
    ax.scatter(*(mean_0 + 0.3 * torch.randn(80, 2)).T.numpy(), 
               c='royalblue', alpha=0.4, s=15, label='$p_0$ (ctrl)')
    ax.scatter(*(mean_1 + 0.3 * torch.randn(80, 2)).T.numpy(), 
               c='crimson', alpha=0.4, s=15, label='$p_1$ (pert)')
    
    ax.set_title(f'Campo $v_\\theta(x, t={t_val:.1f})$', fontsize=13, fontweight='bold')
    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-3.5, 4.5)
    ax.legend(fontsize=9, loc='upper left')

plt.suptitle('Il campo vettoriale appreso dalla rete neurale a tempi diversi', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Le frecce puntano dalla regione blu (controllo) verso la regione rossa (perturbato).')
print('Il campo è quasi costante perché il target u_t = x_1 - x_0 non dipende da t!')

---
## 5. Il Problema dell'Accoppiamento Casuale e l'Optimal Transport

Fin qui abbiamo accoppiato `x_0[i]` con `x_1[i]` secondo l'ordine nel batch. Ma questo è **casuale** e produce traiettorie che si incrociano:

```
Accoppiamento casuale:              Accoppiamento OT:
x₀[0] ──────╲──── x₁[0]            x₀[0] ─────────── x₁[1]  (vicino)
              ╲╱
              ╱╲
x₀[1] ──────╱──── x₁[1]            x₀[1] ─────────── x₁[0]  (vicino)
     ↑ si incrociano!                    ↑ NON si incrociano
```

Le traiettorie incrociate creano **ambiguità**: nello stesso punto dello spazio, il modello deve predire velocità in direzioni opposte. Questo confonde l'addestramento.

In [ ]:
# Dimostriamolo visivamente
torch.manual_seed(7)
n_demo = 8
x0_demo = torch.randn(n_demo, 2) * 0.5 + torch.tensor([-2.0, 0.0])
x1_demo = torch.randn(n_demo, 2) * 0.5 + torch.tensor([+2.0, 0.0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CASUALE
ax = axes[0]
for i in range(n_demo):
    ax.plot([x0_demo[i, 0], x1_demo[i, 0]], [x0_demo[i, 1], x1_demo[i, 1]], 
            'gray', alpha=0.6, linewidth=1.5)
ax.scatter(x0_demo[:, 0], x0_demo[:, 1], c='royalblue', s=80, zorder=5, label='$x_0$')
ax.scatter(x1_demo[:, 0], x1_demo[:, 1], c='crimson', s=80, zorder=5, label='$x_1$')
ax.set_title('Accoppiamento CASUALE\n(le traiettorie si incrociano)', fontsize=12, fontweight='bold')
ax.legend()

# OT
cost = torch.cdist(x0_demo, x1_demo, p=2) ** 2
row_ind, col_ind = linear_sum_assignment(cost.numpy())
x1_ot = x1_demo[col_ind]

ax = axes[1]
for i in range(n_demo):
    ax.plot([x0_demo[i, 0], x1_ot[i, 0]], [x0_demo[i, 1], x1_ot[i, 1]], 
            'gray', alpha=0.6, linewidth=1.5)
ax.scatter(x0_demo[:, 0], x0_demo[:, 1], c='royalblue', s=80, zorder=5, label='$x_0$')
ax.scatter(x1_ot[:, 0], x1_ot[:, 1], c='crimson', s=80, zorder=5, label='$x_1$ (riordinato)')
ax.set_title('Accoppiamento OPTIMAL TRANSPORT\n(traiettorie rettilinee, non incrociate)', fontsize=12, fontweight='bold')
ax.legend()

for ax in axes:
    ax.set_xlim(-4, 4)
    ax.set_ylim(-2.5, 2.5)

plt.tight_layout()
plt.show()

cost_random = torch.sum((x0_demo - x1_demo)**2).item()
cost_ot = torch.sum((x0_demo - x1_ot)**2).item()
print(f'Costo totale casuale: {cost_random:.2f}')
print(f'Costo totale OT:     {cost_ot:.2f}  ({cost_ot/cost_random*100:.0f}% del casuale)')

### 5.1 Scriviamo la classe `OptimalTransportCFM`

Eredita da `ConditionalFlowMatcher` e sovrascrive un solo metodo: inserisce il matching OT **prima** dell'interpolazione.

In [ ]:
class OptimalTransportCFM(ConditionalFlowMatcher):
    """
    OT-CFM: Risolve l'assegnamento ottimo nel mini-batch prima di interpolare.
    
    min_π Σ_i ||x_0[i] - x_1[π(i)]||²
    """
    
    def __init__(self, sigma_min: float = 1e-4):
        super().__init__(sigma_min=sigma_min)

    @torch.no_grad()
    def compute_ot_coupling(self, x_0, x_1):
        """Risolve il matching OT con l'algoritmo ungherese O(B³)."""
        # Matrice di costo: distanze euclidee al quadrato [B, B]
        cost_matrix = torch.cdist(x_0, x_1, p=2) ** 2
        cost_matrix_np = cost_matrix.detach().cpu().numpy()
        
        # Algoritmo ungherese: trova la permutazione ottimale
        row_ind, col_ind = linear_sum_assignment(cost_matrix_np)
        
        return x_0[row_ind], x_1[col_ind]

    def sample_location_and_target(self, x_0, x_1, t=None):
        """Prima il matching OT, poi delega all'interpolazione della classe padre."""
        x_0_ot, x_1_ot = self.compute_ot_coupling(x_0, x_1)
        return super().sample_location_and_target(x_0_ot, x_1_ot, t=t)

print('✓ OptimalTransportCFM definito')
print(f'  Eredita da: {OptimalTransportCFM.__bases__}')
print(f'  Metodi propri: compute_ot_coupling, sample_location_and_target (override)')

### 5.2 Verifichiamo che il coupling OT riduce il costo

In [ ]:
torch.manual_seed(42)
ot_cfm = OptimalTransportCFM(sigma_min=0.0)

B, D = 64, 5
x0 = torch.randn(B, D)
x1 = torch.randn(B, D) + 2.0

# Costo con accoppiamento casuale
cost_random = torch.mean(torch.sum((x0 - x1)**2, dim=-1)).item()

# Costo dopo matching OT
x0_ot, x1_ot = ot_cfm.compute_ot_coupling(x0, x1)
cost_ot = torch.mean(torch.sum((x0_ot - x1_ot)**2, dim=-1)).item()

print(f'Costo medio casuale:  {cost_random:.4f}')
print(f'Costo medio OT:       {cost_ot:.4f}')
print(f'Riduzione:            {(1 - cost_ot/cost_random)*100:.1f}%')
assert cost_ot <= cost_random, 'ERRORE: il costo OT deve essere <= al casuale!'
print('\n✓ Il matching OT riduce sempre il costo di trasporto!')

---
## 6. Test Finale: Addestramento + Generazione con OT-CFM

Addestriamo con OT-CFM e poi **generiamo** nuove cellule perturbate integrando l'ODE appresa con Runge-Kutta 4.

Usiamo il solver dal modulo `cfm/solver.py` (che studieremo nella Lezione 2).

In [ ]:
from cfm.solver import ODESolver

torch.manual_seed(42)
model = ToyVectorField(dim=2, hidden=64)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
ot_cfm = OptimalTransportCFM(sigma_min=0.01)

mean_0 = torch.tensor([-2.0, -1.0])
mean_1 = torch.tensor([+2.0, +1.5])

# Addestramento
for step in range(300):
    optimizer.zero_grad()
    x0 = mean_0 + 0.2 * torch.randn(128, 2)
    x1 = mean_1 + 0.2 * torch.randn(128, 2)
    loss, _ = ot_cfm.compute_loss(model, x0, x1)
    loss.backward()
    optimizer.step()

print(f'Loss finale: {loss.item():.4f}')

# Generazione: integriamo l'ODE da t=0 a t=1
solver = ODESolver(method='rk4', num_steps=20)
test_x0 = mean_0 + 0.2 * torch.randn(200, 2)
x_gen, trajectory = solver.sample(model, test_x0, return_trajectory=True)

# Visualizzazione
fig, ax = plt.subplots(figsize=(8, 6))

# Traiettorie di alcune cellule
traj_np = [s.numpy() for s in trajectory]
for i in range(20):
    line = np.array([t[i] for t in traj_np])
    ax.plot(line[:, 0], line[:, 1], color='gray', alpha=0.3, linewidth=0.8)

ax.scatter(*test_x0.numpy().T, c='royalblue', alpha=0.5, s=20, label='Controllo $x_0$ (t=0)')
ax.scatter(*x_gen.numpy().T, c='crimson', alpha=0.5, s=20, label='Generato $\hat{x}_1$ (t=1)')

# Ground truth
gt = mean_1 + 0.2 * torch.randn(200, 2)
ax.scatter(*gt.numpy().T, c='limegreen', alpha=0.3, s=15, marker='^', label='Ground truth $x_1$')

ax.set_title('OT-CFM: Generazione di cellule perturbate via integrazione ODE (RK4)', 
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

gen_mean = x_gen.mean(dim=0)
print(f'\nMedia generata:     [{gen_mean[0]:.2f}, {gen_mean[1]:.2f}]')
print(f'Media attesa (x₁):  [{mean_1[0]:.2f}, {mean_1[1]:.2f}]')
print(f'Errore:             {torch.norm(gen_mean - mean_1).item():.4f}')

---
## 7. Riepilogo

Cosa abbiamo imparato in questo notebook:

| Concetto | Dove nel codice | Equazione |
|----------|-----------------|----------|
| Interpolazione lineare | `mu_t = (1-t)*x_0 + t*x_1` | $\mu_t = (1-t)x_0 + tx_1$ |
| Rumore regolarizzante | `x_t = mu_t + sigma_min * noise` | $x_t = \mu_t + \sigma_{\min}\epsilon$ |
| Target analitico | `u_t = x_1 - x_0` | $u_t = x_1 - x_0$ (costante!) |
| Loss di regressione | `mean((v_pred - u_t)**2)` | $\|v_\theta - u_t\|^2$ |
| OT mini-batch | `linear_sum_assignment(cost)` | $\min_\pi \sum \|x_0 - x_{1,\pi}\|^2$ |

**Punti chiave:**
1. Il training è una **regressione L2** su un target noto. Nessuna simulazione, nessuna ODE.
2. L'**Optimal Transport** deambigua le traiettorie e le rende rettilinee.
3. Il tempo $t \in [0, 1]$ **non è** tempo biologico. È un parametro di trasformazione distribuzionale.
4. `sigma_min` regolarizza il path senza alterare il target.

### Prossimo notebook
**Lezione 2: `cfm/solver.py`** — L'integratore ODE che genera nuove cellule perturbate al test-time.